<a href="https://colab.research.google.com/github/Ponnaganivamshicharmesh/Employee-Policy-Copilot/blob/main/notebook/Employee_Policy_Copilot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================
# Cell 1: Install packages
# =========================================
!pip install -q rank_bm25 sentence-transformers sentencepiece transformers torch accelerate peft bitsandbytes jinja2 faiss-gpu chromadb langchain langchain-community langchainhub datasets scikit-learn tqdm pandas numpy requests


In [ ]:
# =========================================
# Cell 2: Imports and setup
# =========================================
import os
import json
import glob
import re
import requests
from typing import List, Dict, Tuple
from google.colab import userdata
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PROJECT_ROOT = "/content/drive/MyDrive/Employee_Policy_Copilot_V2"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
DOCUMENTS_DIR = os.path.join(DATA_DIR, "documents")
METADATA_FILE = os.path.join(DATA_DIR, "metadata.json")
EVAL_QA_FILE = os.path.join(DATA_DIR, "evaluation_qa.json")

print(f"Project root exists: {os.path.exists(PROJECT_ROOT)}")
print(f"Documents dir exists: {os.path.exists(DOCUMENTS_DIR)}")
print(f"Metadata file exists: {os.path.exists(METADATA_FILE)}")
print(f"Evaluation QA file exists: {os.path.exists(EVAL_QA_FILE)}")

class Document:
    def __init__(self, content: str, metadata: Dict = None):
        self.content = content
        self.metadata = metadata or {}

In [ ]:
# =========================================
# Cell 3: Load documents
# =========================================
def load_documents_from_folder(folder_path: str) -> List[Document]:
    documents = []
    txt_files = glob.glob(os.path.join(folder_path, "**", "*.txt"), recursive=True)
    print(f"Found {len(txt_files)} .txt files in {folder_path}")

    for txt_file in tqdm(txt_files, desc="Loading files"):
        with open(txt_file, "r", encoding="utf-8") as f:
            content = f.read()

        file_name = os.path.basename(txt_file)
        folder_name = os.path.basename(os.path.dirname(txt_file))

        documents.append(
            Document(
                content=content,
                metadata={
                    "file_name": file_name,
                    "folder": folder_name,
                    "file_path": txt_file,
                    "content_length": len(content)
                }
            )
        )
    return documents

all_documents = load_documents_from_folder(DOCUMENTS_DIR)
print(f"Total documents loaded: {len(all_documents)}")

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)
print(f"Metadata loaded: {len(metadata)} items")

with open(EVAL_QA_FILE, "r", encoding="utf-8") as f:
    evaluation_qa = json.load(f)
print(f"Evaluation QA loaded: {len(evaluation_qa)} questions")

In [ ]:
# =========================================
# Cell 4: Chunk documents
# =========================================
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

def chunk_document(content: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    tokens = content.split()
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunks.append(" ".join(tokens[i:i + chunk_size]))
    return chunks

chunked_documents = []
for doc in all_documents:
    chunks = chunk_document(doc.content)
    for chunk_content in chunks:
        chunked_documents.append(
            Document(
                content=chunk_content,
                metadata={**doc.metadata}
            )
        )

print(f"Total chunks created: {len(chunked_documents)}")

In [ ]:
# =========================================
# Cell 5: Dense embeddings
# =========================================
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
chunk_contents = [chunk.content for chunk in chunked_documents]

dense_embeddings = embedding_model.encode(
    chunk_contents,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True
)

for i, chunk_doc in enumerate(chunked_documents):
    chunk_doc.dense_embedding = dense_embeddings[i]

print(f"Dense embeddings shape: {dense_embeddings.shape}")

In [ ]:
# =========================================
# Cell 6: BM25 index
# =========================================
bm25_tokenized = [chunk.content.split() for chunk in chunked_documents]
bm25_index = BM25Okapi(bm25_tokenized)

for i, chunk_doc in enumerate(chunked_documents):
    chunk_doc.bm25_index_id = i

print("BM25 index ready")

In [ ]:
# =========================================
# Cell 7: Hybrid retrieval
# =========================================
def hybrid_retrieve(query: str,
                    chunked_documents: List[Document],
                    bm25_index: BM25Okapi,
                    embedding_model: SentenceTransformer,
                    top_k: int = 5,
                    bm25_weight: float = 0.5,
                    dense_weight: float = 0.5) -> List[Tuple[Document, float]]:

    query_tokens = query.split()
    bm25_scores = bm25_index.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_rank = {idx: rank + 1 for rank, idx in enumerate(bm25_top_indices) if bm25_scores[idx] > 0}

    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    chunk_embeddings = np.array([doc.dense_embedding for doc in chunked_documents])
    dense_similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    dense_top_indices = np.argsort(dense_similarities)[-top_k:][::-1]
    dense_rank = {idx: rank + 1 for rank, idx in enumerate(dense_top_indices)}

    rrf_k = 60
    combined_scores = {}
    for idx in set(bm25_rank.keys()) | set(dense_rank.keys()):
        bm25_score = 1 / (rrf_k + bm25_rank[idx]) if idx in bm25_rank else 0
        dense_score = 1 / (rrf_k + dense_rank[idx]) if idx in dense_rank else 0
        combined_scores[idx] = bm25_weight * bm25_score + dense_weight * dense_score

    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)[:top_k]
    return [(chunked_documents[idx], combined_scores[idx]) for idx in sorted_indices]

In [ ]:
# =========================================
# Cell 8: Query router
# =========================================
class QueryRouter:
    def __init__(self, chunked_documents: List[Document]):
        self.department_keywords = {
            "hr": ["hr", "human resources", "leave", "attendance", "performance"],
            "engineering": ["engineering", "developer", "code", "secure", "technical"],
            "finance": ["finance", "travel", "reimbursement", "expense", "procurement", "meal"],
            "it": ["it", "password", "mfa", "access", "software", "device"],
            "compliance": ["compliance", "privacy", "incident", "data retention", "acceptable use"]
        }

    def classify_query(self, query: str) -> str:
        q = query.lower()
        for department, keywords in self.department_keywords.items():
            if any(keyword in q for keyword in keywords):
                return f"department_{department}"
        if re.search(r'\d+\.\d+', query) or "version" in q:
            return "keyword_heavy"
        return "general"

    def route(self, query: str) -> dict:
        category = self.classify_query(query)
        if category.startswith("department_"):
            dept = category.split("_")[1]
            return {
                "strategy": "filtered_hybrid",
                "filters": {"department": dept},
                "bm25_weight": 0.5,
                "dense_weight": 0.5,
                "description": f"Filtering for {dept} department policies"
            }
        if category == "keyword_heavy":
            return {
                "strategy": "bm25_heavy",
                "filters": {},
                "bm25_weight": 0.8,
                "dense_weight": 0.2,
                "description": "BM25-heavy retrieval for keyword-specific queries"
            }
        return {
            "strategy": "full_hybrid",
            "filters": {},
            "bm25_weight": 0.5,
            "dense_weight": 0.5,
            "description": "Full hybrid retrieval"
        }

query_router = QueryRouter(chunked_documents)
print("QueryRouter initialized")

In [ ]:
# Cell 9: Query rewriting with NVIDIA NIM
class QueryRewriter:
    def __init__(self, model_name="meta/llama-3.3-70b-instruct", api_key=None):
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("NVIDIA_API_KEY", "")
        self.api_url = "https://integrate.api.nvidia.com/v1/chat/completions"

    def rewrite(self, query: str) -> str:
        prompt = f"""Rewrite the user query for policy retrieval.
Keep the meaning the same.
Make it short, specific, and retrieval-friendly.

User query: {query}
Rewritten query:"""

        response = requests.post(
            self.api_url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": self.model_name,
                "messages": [
                    {"role": "system", "content": "You rewrite user queries for document retrieval."},
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.0,
                "top_p": 1.0,
                "max_tokens": 64,
                "stream": False
            },
            timeout=120
        )

        if response.status_code == 200:
            data = response.json()
            text = data["choices"][0]["message"]["content"].strip()
            return text.split("\n")[0].strip()
        return query

query_rewriter = QueryRewriter()
query_rewriting_enabled = True

def apply_query_rewriting(query: str, enabled: bool = False) -> str:
    if enabled:
        rewritten = query_rewriter.rewrite(query)
        print(f"Original query:  {query}")
        print(f"Rewritten query: {rewritten}")
        return rewritten
    return query

In [ ]:
# Cell 10: Reranker fallback without HuggingFaceCrossEncoder
class Reranker:
    def __init__(self):
        pass

    def rerank(self, query: str, documents: List[Tuple[Document, float]], top_k: int = 3) -> List[Tuple[Document, float]]:
        if not documents:
            return []

        q = set(query.lower().split())
        reranked = []

        for doc, orig_score in documents:
            d = set(doc.content.lower().split())
            overlap = len(q & d)
            length_penalty = min(len(doc.content.split()) / 500.0, 1.0)
            score = 0.7 * float(orig_score) + 0.3 * (overlap / (len(q) + 1e-6)) - 0.05 * length_penalty
            reranked.append((doc, score))

        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked[:top_k]

reranker = Reranker()
print("Reranker initialized with local fallback scoring.")

In [ ]:
# =========================================
# Cell 11: NVIDIA NIM answer generator
# =========================================
class NVIDIAAnswerGenerator:
    def __init__(self, model_name="meta/llama-3.3-70b-instruct", api_key=None):
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("NVIDIA_API_KEY", "")
        self.api_url = "https://integrate.api.nvidia.com/v1/chat/completions"

    def generate(self, query, documents, max_length=500):
        if not documents:
            return "No relevant documents found for your query."

        context = "\n\n".join([doc_score_tuple[0].content for doc_score_tuple in documents[:3]])

        prompt = f"""Answer the user's question using only the policy context below.
If the context does not contain the answer, say you do not have enough information.

Policy context:
{context}

User question:
{query}
"""

        response = requests.post(
            self.api_url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": self.model_name,
                "messages": [
                    {"role": "system", "content": "You answer only from the provided policy context."},
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.2,
                "top_p": 0.9,
                "max_tokens": max_length,
                "stream": False
            },
            timeout=120
        )

        if response.status_code == 200:
            data = response.json()
            return data["choices"][0]["message"]["content"].strip()
        return f"API error ({response.status_code}): {response.text}"

In [ ]:
# =========================================
# Cell 12: Full copilot pipeline
# =========================================
class EmployeePolicyCopilot:
    def __init__(self,
                 chunked_documents: List[Document],
                 bm25_index: BM25Okapi,
                 embedding_model: SentenceTransformer,
                 query_router: QueryRouter,
                 reranker: Reranker = None,
                 answer_generator: NVIDIAAnswerGenerator = None,
                 query_rewriter = None):
        self.chunked_documents = chunked_documents
        self.bm25_index = bm25_index
        self.embedding_model = embedding_model
        self.router = query_router
        self.reranker = reranker
        self.answer_generator = answer_generator
        self.query_rewriter = query_rewriter

    def query(self,
              user_query: str,
              top_k: int = 5,
              enable_query_rewriting: bool = False,
              enable_reranking: bool = False,
              enable_generation: bool = False) -> dict:

        pipeline_steps = []

        processed_query = user_query
        if enable_query_rewriting and self.query_rewriter is not None:
            processed_query = self.query_rewriter.rewrite(user_query)
            pipeline_steps.append("Query rewriting: enabled")
        else:
            pipeline_steps.append("Query rewriting: disabled")

        route = self.router.route(processed_query)
        pipeline_steps.append(f"Query router: {route['strategy']} ({route['description']})")

        results = hybrid_retrieve(
            query=processed_query,
            chunked_documents=self.chunked_documents,
            bm25_index=self.bm25_index,
            embedding_model=self.embedding_model,
            top_k=top_k,
            bm25_weight=route.get("bm25_weight", 0.5),
            dense_weight=route.get("dense_weight", 0.5)
        )
        pipeline_steps.append(f"Hybrid retrieval: {len(results)} documents retrieved")

        final_docs = results[:top_k]
        if enable_reranking and self.reranker is not None:
            final_docs = self.reranker.rerank(processed_query, final_docs, top_k=top_k)
            pipeline_steps.append("Reranking: enabled")
        else:
            pipeline_steps.append("Reranking: disabled")

        answer = None
        if enable_generation and self.answer_generator is not None:
            answer = self.answer_generator.generate(processed_query, final_docs)
            pipeline_steps.append("Answer generation: enabled")
        else:
            pipeline_steps.append("Answer generation: disabled")

        return {
            "query": user_query,
            "processed_query": processed_query,
            "retrieved_docs": final_docs,
            "answer": answer,
            "pipeline_steps": pipeline_steps
        }

In [ ]:
# =========================================
# Cell 13: Evaluation
# =========================================
def evaluate_rag(copilot, evaluation_qa: List[dict], top_k: int = 3, enable_query_rewriting: bool = False) -> dict:
    results = []

    for qa in evaluation_qa:
        query = qa["query"]
        expected_docs = qa.get("retrieval_gt_doc_ids", [])

        result = copilot.query(
            user_query=query,
            top_k=top_k,
            enable_query_rewriting=enable_query_rewriting,
            enable_reranking=False,
            enable_generation=False
        )

        retrieved_pairs = []
        for doc, _ in result["retrieved_docs"]:
            folder = doc.metadata.get("folder", "")
            file_name = doc.metadata.get("file_name", "").replace(".txt", "")
            retrieved_pairs.append(f"{folder}/{file_name}")

        retrieval_hit = int(any(expected in retrieved_pairs for expected in expected_docs))

        results.append({
            "query": query,
            "processed_query": result["processed_query"],
            "retrieved_pairs": retrieved_pairs,
            "expected_docs": expected_docs,
            "retrieval_hit": retrieval_hit
        })

    total_queries = len(results)
    retrieval_hits = sum(r["retrieval_hit"] for r in results)
    retrieval_accuracy = retrieval_hits / total_queries if total_queries else 0

    return {
        "total_queries": total_queries,
        "retrieval_hits": retrieval_hits,
        "retrieval_accuracy": retrieval_accuracy,
        "individual_results": results
    }

In [ ]:
# =========================================
# Cell 14: Initialize copilot
# =========================================
copilot = EmployeePolicyCopilot(
    chunked_documents=chunked_documents,
    bm25_index=bm25_index,
    embedding_model=embedding_model,
    query_router=query_router,
    reranker=reranker,
    answer_generator=None,
    query_rewriter=query_rewriter
)

print("Copilot initialized")

In [ ]:
# =========================================
# Cell 15: Set NVIDIA API and test
# =========================================
from google.colab import userdata
import os

nvidia_key = userdata.get("NVIDIA_API_KEY")
if not nvidia_key:
    raise ValueError("Set NVIDIA_API_KEY in Colab secrets first.")

os.environ["NVIDIA_API_KEY"] = nvidia_key

copilot.answer_generator = NVIDIAAnswerGenerator(
    model_name="meta/llama-3.3-70b-instruct",
    api_key=os.environ.get("NVIDIA_API_KEY", "")
)

test_query = "What is the travel reimbursement policy?"

result = copilot.query(
    user_query=test_query,
    top_k=3,
    enable_query_rewriting=False,
    enable_reranking=True,
    enable_generation=True
)

print("Pipeline steps:")
for step in result["pipeline_steps"]:
    print(" -", step)

print("\nRetrieved documents:")
for i, doc_score_tuple in enumerate(result["retrieved_docs"], start=1):
    doc, score = doc_score_tuple
    print(f"{i}. {doc.metadata['folder']}/{doc.metadata['file_name']} ({score:.4f})")

print("\nGenerated answer:")
print(result["answer"])

In [ ]:
# =========================================
# Cell 16: Run evaluation
# =========================================
eval_off = evaluate_rag(copilot, evaluation_qa, top_k=3, enable_query_rewriting=False)
eval_on = evaluate_rag(copilot, evaluation_qa, top_k=3, enable_query_rewriting=True)

print("Evaluation results")
print(f"Rewriting off: {eval_off['retrieval_hits']}/{eval_off['total_queries']} = {eval_off['retrieval_accuracy']:.2%}")
print(f"Rewriting on : {eval_on['retrieval_hits']}/{eval_on['total_queries']} = {eval_on['retrieval_accuracy']:.2%}")

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

def save_evaluation_outputs(eval_result, output_dir="outputs"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    rows = []
    for item in eval_result["individual_results"]:
        rows.append({
            "query": item.get("query", ""),
            "processed_query": item.get("processed_query", ""),
            "retrieved_pairs": " | ".join(item.get("retrieved_pairs", [])),
            "expected_docs": " | ".join(item.get("expected_docs", [])),
            "retrieval_hit": item.get("retrieval_hit", 0),
            "answer": item.get("answer", "")
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_path / "evaluation_results.csv", index=False)

    summary = {
        "total_queries": eval_result.get("total_queries", 0),
        "retrieval_hits": eval_result.get("retrieval_hits", 0),
        "retrieval_accuracy": eval_result.get("retrieval_accuracy", 0),
    }
    with open(output_path / "evaluation_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    sample_lines = []
    for item in eval_result["individual_results"][:5]:
        sample_lines.append(f"Q: {item.get('query', '')}")
        sample_lines.append(f"Retrieved: {item.get('retrieved_pairs', [])}")
        sample_lines.append(f"Expected: {item.get('expected_docs', [])}")
        if item.get("answer"):
            sample_lines.append(f"Answer: {item.get('answer')}")
        sample_lines.append("")

    with open(output_path / "sample_answers.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(sample_lines))

    return str(output_path)

def evaluate_rag_with_answers(copilot, evaluation_qa, top_k=3, enable_query_rewriting=False):
    results = []
    for qa in evaluation_qa:
        query = qa["query"]
        expected_docs = qa.get("retrieval_gt_doc_ids", [])

        result = copilot.query(
            user_query=query,
            top_k=top_k,
            enable_query_rewriting=enable_query_rewriting,
            enable_reranking=True,
            enable_generation=True,
        )

        retrieved_pairs = []
        for doc, _ in result["retrieved_docs"]:
            folder = doc.metadata.get("folder", "")
            file_name = doc.metadata.get("file_name", "").replace(".txt", "")
            retrieved_pairs.append(f"{folder}/{file_name}")

        retrieval_hit = int(any(expected in retrieved_pairs for expected in expected_docs))

        results.append({
            "query": query,
            "processed_query": result["processed_query"],
            "retrieved_pairs": retrieved_pairs,
            "expected_docs": expected_docs,
            "retrieval_hit": retrieval_hit,
            "answer": result.get("answer", ""),
        })

    total_queries = len(results)
    retrieval_hits = sum(r["retrieval_hit"] for r in results)
    retrieval_accuracy = retrieval_hits / total_queries if total_queries else 0

    return {
        "total_queries": total_queries,
        "retrieval_hits": retrieval_hits,
        "retrieval_accuracy": retrieval_accuracy,
        "individual_results": results,
    }

eval_off_answers = evaluate_rag_with_answers(
    copilot,
    evaluation_qa,
    top_k=3,
    enable_query_rewriting=False
)

eval_on_answers = evaluate_rag_with_answers(
    copilot,
    evaluation_qa,
    top_k=3,
    enable_query_rewriting=True
)

save_evaluation_outputs(eval_off_answers, output_dir=os.path.join(OUTPUT_ROOT, "rewrite_off"))
save_evaluation_outputs(eval_on_answers, output_dir=os.path.join(OUTPUT_ROOT, "rewrite_on"))

print("Saved outputs to:", OUTPUT_ROOT)
print(f"Rewriting off: {eval_off_answers['retrieval_hits']}/{eval_off_answers['total_queries']} = {eval_off_answers['retrieval_accuracy']:.2%}")
print(f"Rewriting on : {eval_on_answers['retrieval_hits']}/{eval_on_answers['total_queries']} = {eval_on_answers['retrieval_accuracy']:.2%}")